In [ ]:
import pandas as pd

appointments = pd.read_csv('/data/appointments.csv')
hospitals = pd.read_csv('/data/hospitals.csv')
patients = pd.read_csv('/data/patients.csv')
doctors = pd.read_csv('/data/doctors.csv')
sms_reminders = pd.read_csv('/data/sms_reminders.csv')
appointments.head()

In [ ]:
hospitals.shape[0]

In [ ]:
doctors.shape[0]

In [ ]:
patients.shape[0]

In [ ]:
doctors['department'].nunique()

In [ ]:
appointments['appointment_id'].nunique()

In [ ]:
appointments.duplicated().sum()

In [ ]:
appointments['status'].nunique()

In [ ]:
# 1. Remove duplicate rows
appointments = appointments.drop_duplicates().copy()


In [ ]:
print(appointments.columns.tolist())

In [ ]:
cleaned = appointments.drop_duplicates().copy()
count = (
    cleaned['fee'].astype(str)
    .str.contains(r'₹|Rs', case=False, na=False)
    .sum()
)

print(count)


In [ ]:

# 2. Standardise status
appointments['status'] = appointments['status'].str.lower().str.strip()
appointments['status'] = appointments['status'].str.replace(' ', '', regex=False)
appointments['status'] = appointments['status'].str.replace('-', '', regex=False)

# 2. Group and count
answer = (
    appointments.groupby('status')
    .size()
    .reset_index(name='appointments')
)

# 3. Display answer
answer

In [ ]:
cleaned = appointments.drop_duplicates().copy()

In [ ]:
# 2. Keep only digits from fee
cleaned['fee'] = (
    cleaned['fee'].astype(str)
    .str.replace(r'\D', '', regex=True)
)

# 3. Convert fee to numeric
cleaned['fee'] = pd.to_numeric(cleaned['fee'], errors='coerce')

# 4. Filter completed appointments
completed = cleaned[cleaned['status'].str.lower().str.strip() == 'completed']

# 5. Calculate total fee
total_fee = completed['fee'].sum()

print(total_fee)

In [ ]:
answer = patients['age'].isna().sum()

answer

In [ ]:
patients['gender'].nunique()

In [ ]:
# 1. Standardise gender
patients['gender'] = patients['gender'].str.strip().str.lower()

patients['gender'] = patients['gender'].replace({
    'm': 'Male',
    'male': 'Male',
    'f': 'Female',
    'female': 'Female'
})

# 2. Group and count
answer = (
    patients.groupby('gender')
    .size()
    .reset_index(name='patients')
)

# 3. Return required columns
answer[['gender', 'patients']]

In [ ]:
# Standardise city names
patients['city'] = patients['city'].str.strip().replace({
    'Aurangabad': 'Chhatrapati Sambhajinagar',
    'Chhatrapati Sambhajinagar': 'Chhatrapati Sambhajinagar'
})

# Count patients
answer = (
    patients[patients['city'] == 'Chhatrapati Sambhajinagar']
    .shape[0]
)

answer

In [ ]:
answer = (
    doctors.groupby('department')
    .size()
    .reset_index(name='doctors')
)

answer[['department', 'doctors']]

In [ ]:
# Remove duplicate rows
cleaned = appointments.drop_duplicates().copy()


In [ ]:

# Group and count
answer = (
    cleaned.groupby('booking_channel')
    .size()
    .reset_index(name='appointments')
)

answer[['booking_channel', 'appointments']]

In [ ]:
answer = (
    cleaned.groupby('payment_mode')
    .size()
    .reset_index(name='appointments')
)

answer[['payment_mode', 'appointments']]

In [ ]:
round(doctors['consultation_fee'].mean(), 0)

In [ ]:
appointments['appointment_date'] = pd.to_datetime(appointments['appointment_date'])

answer = pd.DataFrame({
    'first_date': [appointments['appointment_date'].min()],
    'last_date': [appointments['appointment_date'].max()]
})

answer

In [ ]:
cleaned['appointment_date'] = pd.to_datetime(cleaned['appointment_date'])

# Filter Q3 (1 July to 30 September 2026)
q3 = cleaned[
    (cleaned['appointment_date'] >= '2026-07-01') &
    (cleaned['appointment_date'] <= '2026-09-30')
]

# Total appointments
answer = len(q3)

answer

In [ ]:
answer = (
    cleaned.groupby('visit_type')
    .size()
    .reset_index(name='appointments')
)

answer[['visit_type', 'appointments']]

In [ ]:
sms_reminders.shape[0]

In [ ]:
answer = (
    sms_reminders.groupby('delivery_status')
    .size()
    .reset_index(name='messages')
)

answer[['delivery_status', 'messages']]

In [ ]:
# Standardise status
cleaned['status'] = cleaned['status'].str.lower().str.strip()
cleaned['status'] = cleaned['status'].str.replace(' ', '', regex=False)
cleaned['status'] = cleaned['status'].str.replace('-', '', regex=False)

# Count completed and no-show
completed = (cleaned['status'] == 'completed').sum()
noshow = (cleaned['status'] == 'noshow').sum()

# Calculate No-Show Rate
answer = round((noshow / (completed + noshow)) * 100, 2)

answer

In [ ]:
# Convert date
cleaned['appointment_date'] = pd.to_datetime(cleaned['appointment_date'])

# Filter Q1 and no-show
q1 = cleaned[
    (cleaned['appointment_date'] >= '2026-04-01') &
    (cleaned['appointment_date'] <= '2026-06-30') &
    (cleaned['status'].str.lower().str.strip().str.replace(' ', '', regex=False).str.replace('-', '', regex=False) == 'noshow')
]

# Count no-shows
answer = len(q1)

answer

In [ ]:
# 2. Convert dates
cleaned['appointment_date'] = pd.to_datetime(cleaned['appointment_date'])
cleaned['booking_date'] = pd.to_datetime(cleaned['booking_date'])

# 3. Exclude booking dates after appointment dates
cleaned = cleaned[
    cleaned['booking_date'] <= cleaned['appointment_date']
].copy()

# 4. Calculate lead time in days
cleaned['lead_time'] = (
    cleaned['appointment_date'] - cleaned['booking_date']
).dt.total_seconds() / 86400

# 5. Calculate average and round to 2 decimals
answer = round(cleaned['lead_time'].mean(), 2)

answer

In [ ]:
cleaned = appointments.drop_duplicates().copy()

In [ ]:
# Convert date
cleaned['appointment_date'] = pd.to_datetime(
    cleaned['appointment_date'], errors='coerce'
)

# Keep required date range
cleaned = cleaned[
    cleaned['appointment_date'].between(
        '2026-01-01', '2026-09-30'
    )
].copy()

# Extract month
cleaned['month'] = cleaned['appointment_date'].dt.strftime('%Y-%m')

# Group and count
answer = (
    cleaned.groupby('month')
    .size()
    .reset_index(name='appointments')
    .sort_values('month')
    .reset_index(drop=True))

answer

In [ ]:

# 2. Convert appointment date
cleaned['appointment_date'] = pd.to_datetime(cleaned['appointment_date'])

# 3. Extract full weekday name
cleaned['weekday'] = cleaned['appointment_date'].dt.day_name()

# 4. Group and count
answer = (
    cleaned.groupby('weekday')
    .size()
    .reset_index(name='appointments')
)

# 5. Return required columns
answer[['weekday', 'appointments']]

In [ ]:
cleaned = hospitals.drop_duplicates().copy()
cleaned = appointments.drop_duplicates().copy()


In [ ]:
# 2. Join appointments with doctors
merged = cleaned.merge(
    doctors[['doctor_id', 'hospital_id']],
    on='doctor_id',
    how='left'
)

# 3. Join with hospitals
merged = merged.merge(
    hospitals[['hospital_id', 'hospital_name']],
    on='hospital_id',
    how='left'
)

# 4. Count appointments per hospital
answer = (
    merged.groupby('hospital_name')
    .size()
    .reset_index(name='appointments')
)

answer

In [ ]:
hospitals.head()

In [ ]:
# 2. Standardise status
cleaned['status'] = cleaned['status'].str.lower().str.strip()
cleaned['status'] = cleaned['status'].str.replace(' ', '', regex=False)
cleaned['status'] = cleaned['status'].str.replace('-', '', regex=False)

# 3. Convert appointment date
cleaned['appointment_date'] = pd.to_datetime(cleaned['appointment_date'])

# 4. Filter Q1 (April to June)
q1 = cleaned[
    cleaned['appointment_date'].between('2026-04-01', '2026-06-30')
]

# 5. Count completed and no-shows
completed = (q1['status'] == 'completed').sum()
noshow = (q1['status'] == 'noshow').sum()

# 6. Calculate No-Show Rate
answer = round((noshow / (completed + noshow)) * 100, 2)

answer

In [ ]:
# 2. Standardise status
cleaned['status'] = cleaned['status'].str.lower().str.strip()
cleaned['status'] = cleaned['status'].str.replace(' ', '', regex=False)
cleaned['status'] = cleaned['status'].str.replace('-', '', regex=False)

# 3. Convert appointment date
cleaned['appointment_date'] = pd.to_datetime(cleaned['appointment_date'])

# 4. Filter Q2 (July to September)
q2 = cleaned[
    cleaned['appointment_date'].between('2026-07-01', '2026-09-30')
]

# 5. Count completed and no-shows
completed = (q2['status'] == 'completed').sum()
noshow = (q2['status'] == 'noshow').sum()

# 6. Calculate rate
answer = round(
    (noshow / (completed + noshow)) * 100, 2
)
answer

In [ ]:
# 2. Standardise status
cleaned['status'] = cleaned['status'].str.lower().str.strip()
cleaned['status'] = cleaned['status'].str.replace(' ', '', regex=False)
cleaned['status'] = cleaned['status'].str.replace('-', '', regex=False)

# 3. Convert appointment date
cleaned['appointment_date'] = pd.to_datetime(cleaned['appointment_date'])

# 4. Filter Q2
q2 = cleaned[
    cleaned['appointment_date'].between('2026-07-01', '2026-09-30')
].copy()

# 5. Join doctors and hospitals
merged = q2.merge(
    doctors[['doctor_id', 'hospital_id']],
    on='doctor_id',
    how='left'
)

merged = merged.merge(
    hospitals[['hospital_id', 'hospital_name']],
    on='hospital_id',
    how='left'
)

# 6. Calculate hospital-wise no-show percentage
valid = merged[merged['status'].isin(['completed', 'noshow'])]

answer = (
    valid.groupby('hospital_name')
    .apply(lambda x: round(
        (x['status'].eq('noshow').sum() / len(x)) * 100, 2
    ), include_groups=False)
    .reset_index(name='noshow_pct')
)

answer[['hospital_name', 'noshow_pct']]

In [ ]:

# 2. Standardise status
cleaned['status'] = cleaned['status'].str.lower().str.strip()
cleaned['status'] = cleaned['status'].str.replace(' ', '', regex=False)
cleaned['status'] = cleaned['status'].str.replace('-', '', regex=False)

# 3. Convert appointment date
cleaned['appointment_date'] = pd.to_datetime(cleaned['appointment_date'])

# 4. Filter Q2
q2 = cleaned[
    cleaned['appointment_date'].between('2026-07-01', '2026-09-30')
].copy()

# 5. Join doctors to get department
merged = q2.merge(
    doctors[['doctor_id', 'department']],
    on='doctor_id',
    how='left'
)

# 6. Keep completed and no-show
valid = merged[
    merged['status'].isin(['completed', 'noshow'])
].copy()

# 7. Calculate department-wise counts
valid['noshow'] = (valid['status'] == 'noshow').astype(int)

answer = (
    valid.groupby('department')
    .agg(
        total=('status', 'count'),
        noshows=('noshow', 'sum')
    )
    .reset_index()
)

# 8. Calculate percentage
answer['noshow_pct'] = (
    answer['noshows'] / answer['total'] * 100
).round(2)

# 9. Required output
answer[['department', 'noshow_pct']]

In [ ]:
cleaned = appointments.drop_duplicates().copy()

In [ ]:
# Check original appointment data
print("Total rows:", len(appointments))
print("Exact duplicates:", appointments.duplicated().sum())

# Check status values
print(appointments['status'].value_counts(dropna=False))

# Check Q2 date range
appointments['appointment_date'] = pd.to_datetime(
    appointments['appointment_date']
)

q2 = appointments[
    appointments['appointment_date'].between(
        '2026-07-01', '2026-09-30'
    )
].drop_duplicates()

print("Q2 appointments:", len(q2))
print(q2['status'].value_counts())

In [ ]:

# 2. Standardise status
cleaned['status'] = (
    cleaned['status']
    .str.lower()
    .str.strip()
    .str.replace(' ', '', regex=False)
    .str.replace('-', '', regex=False)
)

# 3. Join doctors to get department
merged = cleaned.merge(
    doctors[['doctor_id', 'department']],
    on='doctor_id',
    how='left'
)

# 4. Keep completed and no-show appointments
valid = merged[
    merged['status'].isin(['completed', 'noshow'])
].copy()

# 5. Calculate department-wise no-show rate
valid['noshow'] = (valid['status'] == 'noshow').astype(int)

answer = (
    valid.groupby('department')
    .agg(
        total=('status', 'count'),
        noshows=('noshow', 'sum')
    )
    .assign(
        noshow_pct=lambda x: (
            x['noshows'] / x['total'] * 100
        ).round(2)
    )
    .reset_index()
    .sort_values('noshow_pct', ascending=False)
)

# 6. Return only expected columns
answer = answer[['department', 'noshow_pct']]

answer

In [ ]:

# 2. Standardise status
cleaned['status'] = (
    cleaned['status']
    .str.lower()
    .str.strip()
    .str.replace(' ', '', regex=False)
    .str.replace('-', '', regex=False)
)

# 3. Convert dates
cleaned['booking_date'] = pd.to_datetime(cleaned['booking_date'])
cleaned['appointment_date'] = pd.to_datetime(cleaned['appointment_date'])

# 4. Calculate lead time in calendar days
cleaned['lead_time'] = (
    cleaned['appointment_date'].dt.normalize() -
    cleaned['booking_date'].dt.normalize()
).dt.days

# 5. Remove negative lead times
cleaned = cleaned[cleaned['lead_time'] >= 0].copy()

# 6. Create lead time bands
cleaned['lead_band'] = pd.cut(
    cleaned['lead_time'],
    bins=[-1, 1, 7, 14, float('inf')],
    labels=['0-1', '2-7', '8-14', '15+']
)

# 7. Keep completed and no-show appointments
valid = cleaned[
    cleaned['status'].isin(['completed', 'noshow'])
].copy()

# 8. Group and calculate
answer = (
    valid.groupby('lead_band', observed=False)
    .agg(
        appointments=('status', 'count'),
        noshows=('status', lambda x: (x == 'noshow').sum())
    )
)

answer['noshow_pct'] = (
    answer['noshows'] / answer['appointments'] * 100
).round(2)

# 9. Return exactly required columns
answer = answer.reset_index()

answer = answer[['lead_band', 'appointments', 'noshow_pct']]

answer

In [ ]:
import numpy as np
import pandas as pd
# 2. Standardise status
cleaned['status'] = (
    cleaned['status'].str.lower().str.strip()
    .str.replace(' ', '', regex=False)
    .str.replace('-', '', regex=False)
)

# 3. Convert dates
cleaned['booking_date'] = pd.to_datetime(cleaned['booking_date'])
cleaned['appointment_date'] = pd.to_datetime(cleaned['appointment_date'])

# 4. Calculate lead time first
cleaned['lead_time'] = (
    cleaned['appointment_date'] - cleaned['booking_date']
).dt.days

# 5. Remove negative lead times
cleaned = cleaned[cleaned['lead_time'] >= 0].copy()

# 6. Create bands using conditions
cleaned['lead_band'] = np.select(
    [
        cleaned['lead_time'] <= 1,
        cleaned['lead_time'] <= 7,
        cleaned['lead_time'] <= 14,
        cleaned['lead_time'] >= 15
    ],
    ['0-1', '2-7', '8-14', '15+'],
    default=None
)

# 7. Count all appointments and calculate no-shows
cleaned['noshow'] = (cleaned['status'] == 'noshow').astype(int)
cleaned['valid'] = cleaned['status'].isin(
    ['completed', 'noshow']
).astype(int)

# 8. Group by lead band
answer = (
    cleaned.groupby('lead_band')
    .agg(
        appointments=('status', 'size'),
        noshows=('noshow', 'sum'),
        valid=('valid', 'sum')
    )
    .reset_index()
)

# 9. Calculate no-show rate
answer['noshow_pct'] = (
    answer['noshows'] / answer['valid'] * 100
).round(2)

# 10. Final required columns
answer = answer[
    ['lead_band', 'appointments', 'noshow_pct']
]

answer

In [ ]:

# 2. Standardise status
cleaned['status'] = (
    cleaned['status'].str.lower().str.strip()
    .str.replace(' ', '', regex=False)
    .str.replace('-', '', regex=False)
)

# 3. Keep completed and no-show for rate calculation
valid = cleaned[
    cleaned['status'].isin(['completed', 'noshow'])
].copy()

# 4. Group by payment mode
valid['noshow'] = (valid['status'] == 'noshow').astype(int)

answer = (
    valid.groupby('payment_mode')
    .agg(
        total=('status', 'count'),
        noshows=('noshow', 'sum')
    )
    .reset_index()
)

# 5. Calculate percentage
answer['noshow_pct'] = (
    answer['noshows'] / answer['total'] * 100
).round(2)

# 6. Return required columns
answer = answer[['payment_mode', 'noshow_pct']]

answer

In [ ]:
# 1. Filter Q2 first
q2 = appointments[
    pd.to_datetime(appointments['appointment_date']).between(
        '2026-07-01', '2026-09-30'
    )
].copy()

# 2. Remove duplicate rows
q2 = q2.drop_duplicates().copy()

# 3. Standardise status
q2['status'] = (
    q2['status'].str.lower().str.strip()
    .str.replace(' ', '', regex=False)
    .str.replace('-', '', regex=False)
)

# 4. Calculate no-show indicator
q2['noshow'] = (q2['status'] == 'noshow').astype(int)

q2['valid'] = q2['status'].isin(
    ['completed', 'noshow']
).astype(int)

# 5. Group by booking channel
answer = (
    q2.groupby('booking_channel')
    .agg(
        appointments=('status', 'size'),
        noshows=('noshow', 'sum'),
        valid=('valid', 'sum')
    )
    .reset_index()
)

# 6. Calculate no-show rate
answer['noshow_pct'] = (
    answer['noshows'] / answer['valid'] * 100
).round(2)

# 7. Final output
answer = answer[
    ['booking_channel', 'appointments', 'noshow_pct']
]

answer

In [ ]:
# 1. Filter Q2 first
q2 = appointments[
    pd.to_datetime(appointments['appointment_date']).between(
        '2026-07-01', '2026-09-30'
    )
].copy()

# 2. Remove duplicates
q2 = q2.drop_duplicates().copy()

# 3. Standardise status
q2['status'] = (
    q2['status'].str.lower().str.strip()
    .str.replace(' ', '', regex=False)
    .str.replace('-', '', regex=False)
)

# 4. Keep only no-show appointments
noshow = q2[q2['status'] == 'noshow'].copy()

# 5. Clean fee - keep digits only
noshow['fee'] = (
    noshow['fee'].astype(str)
    .str.replace(r'\D', '', regex=True)
)

noshow['fee'] = pd.to_numeric(noshow['fee'], errors='coerce')

# 6. Calculate lost revenue
answer = noshow['fee'].sum()

answer

In [ ]:

# 2. Standardise status
cleaned['status'] = (
    cleaned['status'].str.lower().str.strip()
    .str.replace(' ', '', regex=False)
    .str.replace('-', '', regex=False)
)

# 3. Convert appointment date
cleaned['appointment_date'] = pd.to_datetime(
    cleaned['appointment_date']
)

# 4. Extract month
cleaned['month'] = cleaned['appointment_date'].dt.strftime('%Y-%m')

# 5. Calculate monthly no-show rate
cleaned['noshow'] = (cleaned['status'] == 'noshow').astype(int)

cleaned['valid'] = cleaned['status'].isin(
    ['completed', 'noshow']
).astype(int)

answer = (
    cleaned.groupby('month')
    .agg(
        noshows=('noshow', 'sum'),
        valid=('valid', 'sum')
    )
    .reset_index()
)

# 6. Calculate percentage
answer['noshow_pct'] = (
    answer['noshows'] / answer['valid'] * 100
).round(2)

# 7. Sort month-wise
answer = answer.sort_values('month').reset_index(drop=True)

# 8. Return required columns
answer[['month', 'noshow_pct']]

In [ ]:
# 1. Remove duplicate appointments
cleaned = appointments.drop_duplicates().copy()

In [ ]:

cleaned['booking_date'] = pd.to_datetime(cleaned['booking_date'])
cleaned['appointment_date'] = pd.to_datetime(cleaned['appointment_date'])

q2 = cleaned[
    cleaned['appointment_date'].between('2026-07-01', '2026-09-30')
].copy()

# 2. Calculate lead time in days
q2['lead_time'] = (
    q2['appointment_date'].dt.normalize() -
    q2['booking_date'].dt.normalize()
).dt.days

# 3. Keep appointments booked at least 1 day ahead
q2 = q2[q2['lead_time'] >= 1].copy()

# 4. Check whether each appointment has at least one Delivered reminder
sms = sms_reminders.copy()

sms['delivery_status'] = (
    sms['delivery_status'].str.strip().str.lower()
)

delivered = (
    sms.groupby('appointment_id')['delivery_status']
    .apply(lambda x: x.eq('delivered').any())
    .reset_index(name='delivered')
)

# 5. Join reminders
q2 = q2.merge(delivered, on='appointment_id', how='left')

q2['delivered'] = q2['delivered'].fillna(False)

# 6. Join doctors and hospitals
merged = q2.merge(
    doctors[['doctor_id', 'hospital_id']],
    on='doctor_id',
    how='left'
)

merged = merged.merge(
    hospitals[['hospital_id', 'hospital_name']],
    on='hospital_id',
    how='left'
)

# 7. Calculate hospital-wise delivered percentage
answer = (
    merged.groupby('hospital_name')
    .agg(
        appointments=('appointment_id', 'nunique'),
        delivered=('delivered', 'sum')
    )
    .reset_index()
)

answer['delivered_pct'] = (
    answer['delivered'] / answer['appointments'] * 100
).round(2)

# 8. Return required columns
answer = answer[['hospital_name', 'delivered_pct']]

answer

In [ ]:

# 2. Filter Q2 first
cleaned['appointment_date'] = pd.to_datetime(cleaned['appointment_date'])
cleaned['booking_date'] = pd.to_datetime(cleaned['booking_date'])

q2 = cleaned[
    cleaned['appointment_date'].between('2026-07-01', '2026-09-30')
].copy()

# 3. Keep appointments booked at least 1 day ahead
q2['lead_time'] = (
    q2['appointment_date'].dt.normalize() -
    q2['booking_date'].dt.normalize()
).dt.days

q2 = q2[q2['lead_time'] >= 1].copy()

# 4. Standardise status
q2['status'] = (
    q2['status'].str.lower().str.strip()
    .str.replace(' ', '', regex=False)
    .str.replace('-', '', regex=False)
)

# 5. Check at least one Delivered reminder per appointment
sms = sms_reminders.copy()
sms['delivery_status'] = sms['delivery_status'].str.strip().str.lower()

delivered = (
    sms.groupby('appointment_id')['delivery_status']
    .apply(lambda x: x.eq('delivered').any())
    .reset_index(name='is_delivered')
)

# 6. Join reminder information
q2 = q2.merge(delivered, on='appointment_id', how='left')

q2['reminder'] = q2['is_delivered'].map({
    True: 'Delivered',
    False: 'Not delivered'
}).fillna('Not delivered')

# 7. Keep completed and no-show for rate calculation
valid = q2[
    q2['status'].isin(['completed', 'noshow'])
].copy()

# 8. Calculate reminder-wise no-show rate
valid['noshow'] = (valid['status'] == 'noshow').astype(int)

answer = (
    valid.groupby('reminder')
    .agg(
        noshows=('noshow', 'sum'),
        total=('status', 'count')
    )
    .reset_index()
)

answer['noshow_pct'] = (
    answer['noshows'] / answer['total'] * 100
).round(2)

# 9. Final output
answer = answer[['reminder', 'noshow_pct']]

answer

In [ ]:

# 2. Standardise appointment status
cleaned['status'] = (
    cleaned['status'].str.lower().str.strip()
    .str.replace(' ', '', regex=False)
    .str.replace('-', '', regex=False)
)

# 3. Join appointments with patients
merged = cleaned.merge(
    patients[['patient_id', 'age']],
    on='patient_id',
    how='inner'
)

# 4. Remove patients with missing age
merged = merged.dropna(subset=['age']).copy()

# 5. Create age bands
merged['age_band'] = pd.cut(
    merged['age'],
    bins=[-1, 14, 30, 45, 59, float('inf')],
    labels=['0-14', '15-30', '31-45', '46-59', '60+']
)

# 6. Keep completed and no-show appointments
valid = merged[
    merged['status'].isin(['completed', 'noshow'])
].copy()

# 7. Calculate no-show rate
valid['noshow'] = (valid['status'] == 'noshow').astype(int)

answer = (
    valid.groupby('age_band', observed=False)
    .agg(
        noshows=('noshow', 'sum'),
        total=('status', 'count')
    )
    .reset_index()
)

answer['noshow_pct'] = (
    answer['noshows'] / answer['total'] * 100
).round(2)

# 8. Final output
answer = answer[['age_band', 'noshow_pct']]

answer

In [ ]:

# 2. Standardise status
cleaned['status'] = (
    cleaned['status'].str.lower().str.strip()
    .str.replace(' ', '', regex=False)
    .str.replace('-', '', regex=False)
)

# 3. Keep completed and no-show appointments
valid = cleaned[
    cleaned['status'].isin(['completed', 'noshow'])
].copy()

# 4. Calculate no-show rate
valid['noshow'] = (valid['status'] == 'noshow').astype(int)

answer = (
    valid.groupby('visit_type')
    .agg(
        noshows=('noshow', 'sum'),
        total=('status', 'count')
    )
    .reset_index()
)

answer['noshow_pct'] = (
    answer['noshows'] / answer['total'] * 100
).round(2)

# 5. Final output
answer = answer[['visit_type', 'noshow_pct']]

answer

In [ ]:

# 2. Convert dates
cleaned['appointment_date'] = pd.to_datetime(cleaned['appointment_date'])
cleaned['booking_date'] = pd.to_datetime(cleaned['booking_date'])

# 3. Filter Q2 first
q2 = cleaned[
    cleaned['appointment_date'].between('2026-07-01', '2026-09-30')
].copy()

# 4. Calculate lead time in days
q2['lead_days'] = (
    q2['appointment_date'] - q2['booking_date']
).dt.days

# 5. Exclude negative lead times
q2 = q2[q2['lead_days'] >= 0].copy()

# 6. Average lead time by booking channel
answer = (
    q2.groupby('booking_channel')['lead_days']
    .mean()
    .round(1)
    .reset_index(name='avg_lead_days')
)

# 7. Final output
answer = answer[['booking_channel', 'avg_lead_days']]

answer

In [ ]:

# 2. Standardise status
cleaned['status'] = (
    cleaned['status'].str.lower().str.strip()
    .str.replace(' ', '', regex=False)
    .str.replace('-', '', regex=False)
)

# 3. Keep only no-show appointments
noshow = cleaned[cleaned['status'] == 'noshow']

# 4. Count no-shows per patient
patient_counts = noshow.groupby('patient_id').size()

# 5. Count patients with 3 or more no-shows
answer = (patient_counts >= 3).sum()

answer

In [ ]:
answer = len(appointments)
answer

In [ ]:


# 2. Convert dates and standardise status
cleaned['appointment_date'] = pd.to_datetime(cleaned['appointment_date'])
cleaned['booking_date'] = pd.to_datetime(cleaned['booking_date'])

cleaned['status'] = (
    cleaned['status'].str.lower().str.strip()
    .str.replace(' ', '', regex=False)
    .str.replace('-', '', regex=False)
)

# 3. Find Nagpur hospital ID through hospitals table
nagpur_ids = hospitals.loc[
    hospitals['city'].str.strip().str.lower() == 'nagpur',
    'hospital_id'
]

# 4. Join appointments to doctors, then filter by hospital_id
nagpur = cleaned.merge(
    doctors[['doctor_id', 'hospital_id']],
    on='doctor_id',
    how='inner'
)

nagpur = nagpur[nagpur['hospital_id'].isin(nagpur_ids)].copy()

# 5. Create month and lead time
nagpur['month'] = nagpur['appointment_date'].dt.to_period('M').astype(str)

nagpur['lead_days'] = (
    nagpur['appointment_date'].dt.normalize() -
    nagpur['booking_date'].dt.normalize()
).dt.days

# 6. Get delivered reminder flag per appointment
sms = sms_reminders.copy()
sms['delivery_status'] = sms['delivery_status'].str.strip().str.lower()

delivered = (
    sms.groupby('appointment_id')['delivery_status']
    .apply(lambda x: x.eq('delivered').any())
    .reset_index(name='is_delivered')
)

nagpur = nagpur.merge(delivered, on='appointment_id', how='left')
nagpur['is_delivered'] = nagpur['is_delivered'].fillna(False)

# 7. Delivered percentage: only appointments booked 1+ days ahead
eligible = nagpur[nagpur['lead_days'] >= 1].copy()

delivery = (
    eligible.groupby('month')
    .agg(
        delivered_count=('is_delivered', 'sum'),
        total=('appointment_id', 'count')
    )
    .reset_index()
)

delivery['delivered_pct'] = (
    delivery['delivered_count'] / delivery['total'] * 100
).round(1)

# 8. No-show percentage: all Nagpur appointments
valid = nagpur[
    nagpur['status'].isin(['completed', 'noshow'])
].copy()

valid['noshow'] = (valid['status'] == 'noshow').astype(int)

no_show = (
    valid.groupby('month')
    .agg(
        noshows=('noshow', 'sum'),
        total=('status', 'count')
    )
    .reset_index()
)

no_show['noshow_pct'] = (
    no_show['noshows'] / no_show['total'] * 100
).round(1)

# 9. Combine results
answer = (
    no_show[['month', 'noshow_pct']]
    .merge(delivery[['month', 'delivered_pct']], on='month', how='left')
)

answer = answer[['month', 'delivered_pct', 'noshow_pct']]
answer = answer.sort_values('month').reset_index(drop=True)

answer

In [ ]:

# 2. Convert dates
cleaned['appointment_date'] = pd.to_datetime(cleaned['appointment_date'])
cleaned['booking_date'] = pd.to_datetime(cleaned['booking_date'])

# 3. Filter Q2 first
q2 = cleaned[
    cleaned['appointment_date'].between('2026-07-01', '2026-09-30')
].copy()

# 4. Calculate lead time
q2['lead_days'] = (
    q2['appointment_date'].dt.normalize() -
    q2['booking_date'].dt.normalize()
).dt.days

# 5. Keep appointments booked 1 to 14 days ahead
q2 = q2[q2['lead_days'].between(1, 14)].copy()

# 6. Standardise status
q2['status'] = (
    q2['status'].str.lower().str.strip()
    .str.replace(' ', '', regex=False)
    .str.replace('-', '', regex=False)
)

# 7. Find appointments with at least one delivered reminder
sms = sms_reminders.copy()
sms['delivery_status'] = sms['delivery_status'].str.strip().str.lower()

delivered = (
    sms.groupby('appointment_id')['delivery_status']
    .apply(lambda x: x.eq('delivered').any())
    .reset_index(name='is_delivered')
)

q2 = q2.merge(delivered, on='appointment_id', how='left')

q2 = q2[q2['is_delivered'].eq(True)].copy()

# 8. Keep completed and no-show appointments
valid = q2[q2['status'].isin(['completed', 'noshow'])].copy()

# 9. Calculate no-show rate
answer = (
    (valid['status'] == 'noshow').sum()
    / len(valid) * 100
).round(2)

answer